# Re-derive the headline results from the released records (CPU, offline, about a minute)

This notebook recomputes the writeup's headline TranscriptLab numbers from the raw generation records of the
registered runs on public PNC states:
* copying, induction, the dose at k = 4 and quarantine removal on Gemma-4-31B 4-bit, and the bf16 row;
* the PNC corpus size.

Each value is compared with the released result that `RESULTS.md` cites. The code uses the standard library only and
imports nothing from the research code.

**On Kaggle:** attach the repository dataset (and the PNC dataset, if it is a separate one). The repository and the
corpus are found under `/kaggle/input/` automatically. **Elsewhere:** run from a checkout, or set `TRANSCRIPTLAB_ROOT`
and `PNC_ROOT`.

In [ ]:
import glob, json, os, subprocess, sys

def find_repo():
    cands = [os.environ.get('TRANSCRIPTLAB_ROOT'), os.getcwd(), *map(str, __import__('pathlib').Path.cwd().parents),
             *sorted(glob.glob('/kaggle/input/*')), *sorted(glob.glob('/kaggle/input/*/*'))]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, 'notebooks', 'recompute.py')) and os.path.isdir(os.path.join(c, 'experiments')):
            return c
    raise SystemExit('repository not found: attach the repository dataset (Kaggle) or set TRANSCRIPTLAB_ROOT')

REPO = find_repo()
OUT = '/kaggle/working/reproduced' if os.path.isdir('/kaggle/working') else os.path.join(os.getcwd(), 'reproduced')
print('repository:', REPO)

In [ ]:
env = dict(os.environ, OUT_DIR=OUT, PYTHONDONTWRITEBYTECODE='1')
r = subprocess.run([sys.executable, os.path.join(REPO, 'notebooks', 'reproduce_numbers.py'), REPO], capture_output=True, text=True, env=env)
print(r.stdout[-6000:]); print(r.stderr[-2000:]); print('exit code', r.returncode)

In [ ]:
l2 = json.load(open(os.path.join(OUT, 'l2_checks.json')))
for row in l2['rows']:
    print(f"{row['quantity']:46s} {row['recomputed']!s:>22s}  released {row['released']!s:>8s}  {row['status']}  <- {row['released_at']}")
print('skipped:', l2['skipped'] or 'none')